# Static Image vs. Dynamic Image Significance Tests

This notebook compares the static and dynamic image-based FER models on the same held-out test samples.

The test set contains **378 reenactments** with two corresponding view-level samples per reenactment:

- one Central-view sample
- one Side-view sample

The Static and Dynamic Image models therefore produce predictions for the same **756 view-level samples**. The pooled accuracy difference is computed over all 756 predictions, while statistical resampling uses the **378 reenactments as paired clusters** so that Central and Side from the same reenactment are not treated as independent observations.

## Analysis plan

1. Primary comparison: compare pooled Dynamic- and Static-Image accuracy using a centered paired cluster bootstrap over reenactments.
2. Sensitivity check: test the same 378 reenactment-level paired differences with a one-sample t-test.
3. Participant-level heterogeneity check: summarize the pooled Dynamic-vs.-Static Image difference separately for each of the eight held-out participants.

### Why a reenactment-level cluster bootstrap is used

At the view level, every Static Image prediction has an exactly corresponding Dynamic Image prediction for the same `sample_id`. A standard paired binary test such as McNemar's test over all 756 views would, however, treat the Central and Side samples from the same reenactment as independent observations.

Instead, the two paired view-level correctness differences are averaged within each reenactment. This yields 378 paired cluster-level differences with support $\{-1,-0.5,0,0.5,1\}$. Their mean is exactly equal to the difference between the two pooled accuracies over all 756 views.

For consistency with the other Image-based comparisons, the primary significance test uses a **centered paired cluster bootstrap** over reenactments. The same resampling scheme provides the 95% confidence interval.

The participant-level analysis is descriptive only. The primary inference concerns the fixed held-out test set and its reenactments rather than population-level generalization across participants.

## 1. Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.stats import ttest_1samp


STATIC_PREDICTIONS_PATH = Path("static_test_predictions.csv")
DYNAMIC_PREDICTIONS_PATH = Path("dynamic_test_predictions.csv")

RANDOM_SEED = 42

# Use many resamples for stable final confidence intervals and p-values.
# Resampling is processed in batches below to keep memory usage low.
N_BOOTSTRAP = 1_000_000
BOOTSTRAP_BATCH_SIZE = 10_000

ALPHA = 0.05

## 2. Load and Validate Predictions

Both prediction CSV files must contain the same 756 `sample_id`s and 378 `reenactment_id`s. For every matched sample, the reenactment, participant, camera view, and ground-truth label must agree exactly between Static and Dynamic predictions.

In [2]:
static_prediction_df = pd.read_csv(STATIC_PREDICTIONS_PATH)
dynamic_prediction_df = pd.read_csv(DYNAMIC_PREDICTIONS_PATH)

print(f"Static rows:  {len(static_prediction_df)}")
print(f"Dynamic rows: {len(dynamic_prediction_df)}")
print(f"Static columns:  {len(static_prediction_df.columns)}")
print(f"Dynamic columns: {len(dynamic_prediction_df.columns)}")

static_prediction_df.head()

Static rows:  756
Dynamic rows: 756
Static columns:  39
Dynamic columns: 39


,sample_id,reenactment_id,timestamp,set_id,participant_id,level_id,emoji_id,camera_index,perspective,true_label_id,...,fea_prob_surprise,multimodal_pred_id,multimodal_pred,multimodal_prob_anger,multimodal_prob_disgust,multimodal_prob_fear,multimodal_prob_happiness,multimodal_prob_neutral,multimodal_prob_sadness,multimodal_prob_surprise
0,1700478995850-2-1-1-0-0-0,1700478995850-2-1-1-0-0,1700478995850,2,1,1,0,0,Central,0,...,0.004558,0,Anger,0.677313,0.030212,0.002842,0.003746,0.026476,0.252719,0.006691
1,1700478995850-2-1-1-0-0-1,1700478995850-2-1-1-0-0,1700478995850,2,1,1,0,1,Side,0,...,0.004558,4,Neutral,0.095327,0.056766,0.039777,0.011597,0.752703,0.013406,0.030424
2,1700478998549-2-1-1-1-5-0,1700478998549-2-1-1-1-5,1700478998549,2,1,1,1,0,Central,5,...,0.000017,5,Sadness,0.000005,0.000117,0.000001,0.000006,0.000001,0.999867,0.000003
3,1700478998549-2-1-1-1-5-1,1700478998549-2-1-1-1-5,1700478998549,2,1,1,1,1,Side,5,...,0.000017,5,Sadness,0.000149,0.000589,0.000038,0.000047,0.000028,0.999063,0.000088
4,1700479001137-2-1-1-2-3-0,1700479001137-2-1-1-2-3,1700479001137,2,1,1,2,0,Central,3,...,0.000797,3,Happiness,0.000376,0.001057,0.000616,0.995559,0.000666,0.000936,0.000790


In [3]:
required_columns = {
    "sample_id",
    "reenactment_id",
    "participant_id",
    "camera_index",
    "true_label_id",
    "image_pred_id"
}

for name, df in [("Static", static_prediction_df), ("Dynamic", dynamic_prediction_df)]:
    missing_columns = required_columns - set(df.columns)
    assert not missing_columns, f"{name}: missing required columns: {sorted(missing_columns)}"

    assert len(df) == 756
    assert df["sample_id"].is_unique
    assert df["reenactment_id"].nunique() == 378
    assert set(df["camera_index"].unique()) == {0, 1}

    samples_per_reenactment = df.groupby("reenactment_id").size()
    assert samples_per_reenactment.eq(2).all()

    views_per_reenactment = df.groupby("reenactment_id")["camera_index"].nunique()
    assert views_per_reenactment.eq(2).all()

    true_labels_per_reenactment = df.groupby("reenactment_id")["true_label_id"].nunique()
    assert true_labels_per_reenactment.eq(1).all()

    participants_per_reenactment = df.groupby("reenactment_id")["participant_id"].nunique()
    assert participants_per_reenactment.eq(1).all()
    assert df["participant_id"].nunique() == 8

assert set(static_prediction_df["sample_id"]) == set(dynamic_prediction_df["sample_id"])
assert set(static_prediction_df["reenactment_id"]) == set(dynamic_prediction_df["reenactment_id"])

static_metadata = (
    static_prediction_df[
        ["sample_id", "reenactment_id", "participant_id", "camera_index", "true_label_id"]
    ]
    .sort_values("sample_id")
    .reset_index(drop=True)
)

dynamic_metadata = (
    dynamic_prediction_df[
        ["sample_id", "reenactment_id", "participant_id", "camera_index", "true_label_id"]
    ]
    .sort_values("sample_id")
    .reset_index(drop=True)
)

pd.testing.assert_frame_equal(static_metadata, dynamic_metadata)

print("Both prediction tables and their cross-setting sample alignment were validated.")

Both prediction tables and their cross-setting sample alignment were validated.


## 3. Construct Reenactment-Level Analysis Table

In [4]:
static_view_df = static_prediction_df[
    ["sample_id", "reenactment_id", "participant_id", "camera_index", "true_label_id", "image_pred_id"]
].copy()

dynamic_view_df = dynamic_prediction_df[
    ["sample_id", "reenactment_id", "participant_id", "camera_index", "true_label_id", "image_pred_id"]
].copy()

paired_view_df = static_view_df.merge(
    dynamic_view_df,
    on="sample_id",
    how="inner",
    suffixes=("_static", "_dynamic"),
    validate="one_to_one"
)

for column in ["reenactment_id", "participant_id", "camera_index", "true_label_id"]:
    assert paired_view_df[f"{column}_static"].equals(paired_view_df[f"{column}_dynamic"])

paired_view_df["static_image_correct"] = (
    paired_view_df["image_pred_id_static"] == paired_view_df["true_label_id_static"]
)

paired_view_df["dynamic_image_correct"] = (
    paired_view_df["image_pred_id_dynamic"] == paired_view_df["true_label_id_static"]
)

assert len(paired_view_df) == 756

paired_view_df.head()

,sample_id,reenactment_id_static,participant_id_static,camera_index_static,true_label_id_static,image_pred_id_static,reenactment_id_dynamic,participant_id_dynamic,camera_index_dynamic,true_label_id_dynamic,image_pred_id_dynamic,static_image_correct,dynamic_image_correct
0,1700478995850-2-1-1-0-0-0,1700478995850-2-1-1-0-0,1,0,0,0,1700478995850-2-1-1-0-0,1,0,0,0,True,True
1,1700478995850-2-1-1-0-0-1,1700478995850-2-1-1-0-0,1,1,0,0,1700478995850-2-1-1-0-0,1,1,0,1,True,False
2,1700478998549-2-1-1-1-5-0,1700478998549-2-1-1-1-5,1,0,5,5,1700478998549-2-1-1-1-5,1,0,5,5,True,True
3,1700478998549-2-1-1-1-5-1,1700478998549-2-1-1-1-5,1,1,5,5,1700478998549-2-1-1-1-5,1,1,5,5,True,True
4,1700479001137-2-1-1-2-3-0,1700479001137-2-1-1-2-3,1,0,3,3,1700479001137-2-1-1-2-3,1,0,3,3,True,True


In [5]:
static_correct_by_view = (
    paired_view_df
    .pivot(index="reenactment_id_static", columns="camera_index_static", values="static_image_correct")
    .rename(columns={
        0: "central_static_image_correct",
        1: "side_static_image_correct"
    })
)

dynamic_correct_by_view = (
    paired_view_df
    .pivot(index="reenactment_id_static", columns="camera_index_static", values="dynamic_image_correct")
    .rename(columns={
        0: "central_dynamic_image_correct",
        1: "side_dynamic_image_correct"
    })
)

participant_id = paired_view_df.groupby("reenactment_id_static")["participant_id_static"].first()
participant_id.name = "participant_id"

analysis_df = static_correct_by_view.join(dynamic_correct_by_view).join(participant_id)

correctness_columns = [
    "central_static_image_correct",
    "side_static_image_correct",
    "central_dynamic_image_correct",
    "side_dynamic_image_correct"
]

analysis_df[correctness_columns] = analysis_df[correctness_columns].astype(bool)

analysis_df["static_image_correct_mean"] = (
    analysis_df["central_static_image_correct"].astype(float)
    + analysis_df["side_static_image_correct"].astype(float)
) / 2.0

analysis_df["dynamic_image_correct_mean"] = (
    analysis_df["central_dynamic_image_correct"].astype(float)
    + analysis_df["side_dynamic_image_correct"].astype(float)
) / 2.0

assert len(analysis_df) == 378
assert not analysis_df.isna().any().any()
assert set(analysis_df["static_image_correct_mean"].unique()).issubset({0.0, 0.5, 1.0})
assert set(analysis_df["dynamic_image_correct_mean"].unique()).issubset({0.0, 0.5, 1.0})

analysis_df.head()

,central_static_image_correct,side_static_image_correct,central_dynamic_image_correct,side_dynamic_image_correct,participant_id,static_image_correct_mean,dynamic_image_correct_mean
reenactment_id_static,,,,,,,
1700478995850-2-1-1-0-0,True,True,True,False,1,1.0,0.5
1700478998549-2-1-1-1-5,True,True,True,True,1,1.0,1.0
1700479001137-2-1-1-2-3,True,True,True,False,1,1.0,0.5
1700479004312-2-1-1-3-0,False,False,False,False,1,0.0,0.0
1700479005401-2-1-1-4-0,True,True,True,True,1,1.0,1.0


## 4. Verify Reported Performance

In [6]:
pooled_static_image_accuracy = analysis_df["static_image_correct_mean"].mean()
pooled_dynamic_image_accuracy = analysis_df["dynamic_image_correct_mean"].mean()

central_static_image_accuracy = analysis_df["central_static_image_correct"].mean()
side_static_image_accuracy = analysis_df["side_static_image_correct"].mean()
central_dynamic_image_accuracy = analysis_df["central_dynamic_image_correct"].mean()
side_dynamic_image_accuracy = analysis_df["side_dynamic_image_correct"].mean()

accuracy_difference = pooled_dynamic_image_accuracy - pooled_static_image_accuracy

print(f"Pooled Static Image accuracy:   {pooled_static_image_accuracy:.4%}")
print(f"Pooled Dynamic Image accuracy:  {pooled_dynamic_image_accuracy:.4%}")
print(f"Central Static Image accuracy:  {central_static_image_accuracy:.4%}")
print(f"Side Static Image accuracy:     {side_static_image_accuracy:.4%}")
print(f"Central Dynamic Image accuracy: {central_dynamic_image_accuracy:.4%}")
print(f"Side Dynamic Image accuracy:    {side_dynamic_image_accuracy:.4%}")
print(f"Dynamic - Static Image:         {100 * accuracy_difference:.2f} percentage points")

assert np.isclose(
    pooled_static_image_accuracy,
    (static_prediction_df["image_pred_id"] == static_prediction_df["true_label_id"]).mean()
)

assert np.isclose(
    pooled_dynamic_image_accuracy,
    (dynamic_prediction_df["image_pred_id"] == dynamic_prediction_df["true_label_id"]).mean()
)

assert int((static_prediction_df["image_pred_id"] == static_prediction_df["true_label_id"]).sum()) == 528
assert np.isclose(pooled_static_image_accuracy, 528 / 756)

assert int((dynamic_prediction_df["image_pred_id"] == dynamic_prediction_df["true_label_id"]).sum()) == 550
assert np.isclose(pooled_dynamic_image_accuracy, 550 / 756)

Pooled Static Image accuracy:   69.8413%
Pooled Dynamic Image accuracy:  72.7513%
Central Static Image accuracy:  72.7513%
Side Static Image accuracy:     66.9312%
Central Dynamic Image accuracy: 73.0159%
Side Dynamic Image accuracy:    72.4868%
Dynamic - Static Image:         2.91 percentage points


## 5. Primary Static Image vs. Dynamic Image Comparison

The primary estimand is the difference between the reported pooled accuracies,

$\Delta = \mathrm{Accuracy}_{Dynamic\ Image} - \mathrm{Accuracy}_{Static\ Image}$.

For each reenactment, the two paired view-level differences are averaged:

$d_i = \big[(D_{i,C} - S_{i,C}) + (D_{i,S} - S_{i,S})\big] / 2$.

The mean of these 378 reenactment-level differences is exactly the difference between the two pooled accuracies over all 756 view samples.

The 378 reenactment-level differences are used as the resampling units.

The analysis estimates:

1. a two-sided bootstrap p-value for $H_0: \Delta = 0$, using the centered empirical distribution under the null
2. a percentile-bootstrap 95% confidence interval for $\Delta$

Positive values favor the Dynamic Image model.

In [7]:
def paired_cluster_bootstrap(differences: np.ndarray,
                             n_bootstrap: int = N_BOOTSTRAP,
                             batch_size: int = BOOTSTRAP_BATCH_SIZE,
                             seed: int = RANDOM_SEED,
                             alpha: float = ALPHA) -> dict:
    
    differences = np.asarray(differences, dtype=float)

    if differences.ndim != 1 or differences.size == 0:
        raise ValueError("differences must be a nonempty one-dimensional array.")
    if not np.isin(differences, [-1.0, -0.5, 0.0, 0.5, 1.0]).all():
        raise ValueError("Expected paired accuracy differences in {-1, -0.5, 0, 0.5, 1}.")

    n = len(differences)
    observed_difference = differences.mean()

    doubled_differences = (2 * differences).astype(np.int64)
    observed_sum = doubled_differences.sum()

    rng = np.random.default_rng(seed)
    bootstrap_means = np.empty(n_bootstrap, dtype=float)
    n_extreme = 0

    for start in range(0, n_bootstrap, batch_size):
        end = min(start + batch_size, n_bootstrap)
        current_batch_size = end - start

        # Ordinary paired bootstrap for the confidence interval.
        bootstrap_indices = rng.integers(0, n, size=(current_batch_size, n))
        bootstrap_means[start:end] = differences[bootstrap_indices].mean(axis=1)

        # Centered-bootstrap null test evaluated in exact integer arithmetic:
        # |mean(d*) - mean(d)| >= |mean(d)| is equivalent to |S* - S| >= |S|,
        # where q_i = 2 d_i, S = sum(q_i), and S* = sum(q_i*).
        null_indices = rng.integers(0, n, size=(current_batch_size, n))
        null_sums = doubled_differences[null_indices].sum(axis=1)

        n_extreme += np.count_nonzero(
            np.abs(null_sums - observed_sum) >= abs(observed_sum)
        )

    ci_low, ci_high = np.quantile(bootstrap_means, [alpha / 2, 1 - alpha / 2])
    p_value = (n_extreme + 1) / (n_bootstrap + 1)

    return {
        "n": n,
        "difference": observed_difference,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "p_value": p_value,
        "n_bootstrap": n_bootstrap
    }


In [8]:
reenactment_differences = (
    analysis_df["dynamic_image_correct_mean"]
    - analysis_df["static_image_correct_mean"]
).to_numpy()

assert np.isclose(reenactment_differences.mean(), accuracy_difference)

primary_result = paired_cluster_bootstrap(
    differences=reenactment_differences,
    n_bootstrap=N_BOOTSTRAP,
    batch_size=BOOTSTRAP_BATCH_SIZE,
    seed=RANDOM_SEED,
    alpha=ALPHA
)

primary_result_df = pd.DataFrame([{
    "comparison": "Pooled Dynamic Image - pooled Static Image",
    "n_reenactments": primary_result["n"],
    "static_image_accuracy": pooled_static_image_accuracy,
    "dynamic_image_accuracy": pooled_dynamic_image_accuracy,
    "difference_pp": 100 * primary_result["difference"],
    "ci_low_pp": 100 * primary_result["ci_low"],
    "ci_high_pp": 100 * primary_result["ci_high"],
    "p_value": primary_result["p_value"],
    "n_bootstrap": primary_result["n_bootstrap"]
}])

primary_result_df

,comparison,n_reenactments,static_image_accuracy,dynamic_image_accuracy,difference_pp,ci_low_pp,ci_high_pp,p_value,n_bootstrap
0,Pooled Dynamic Image - pooled Static Image,378,0.698413,0.727513,2.910053,-0.793651,6.613757,0.129288,1000000


In [9]:
row = primary_result_df.iloc[0]

print(f"Dynamic - Static Image accuracy difference: {row['difference_pp']:.2f} percentage points")
print(f"{100 * (1 - ALPHA):.0f}% bootstrap CI: [{row['ci_low_pp']:.2f}, {row['ci_high_pp']:.2f}] percentage points")
print(f"Two-sided bootstrap p-value: {row['p_value']:.12f}")

Dynamic - Static Image accuracy difference: 2.91 percentage points
95% bootstrap CI: [-0.79, 6.61] percentage points
Two-sided bootstrap p-value: 0.129287870712


## 6. Sensitivity Check

As a sensitivity analysis, apply a one-sample t-test to the same 378 reenactment-level paired differences.

This is not a separate research hypothesis and is not included in a multiplicity-correction family. It checks whether the inferential conclusion agrees with the primary bootstrap analysis.

In [10]:
sensitivity_test = ttest_1samp(reenactment_differences, popmean=0)

sensitivity_result_df = pd.DataFrame([{
    "comparison": "Pooled Dynamic Image - pooled Static Image",
    "n_reenactments": len(reenactment_differences),
    "difference_pp": 100 * reenactment_differences.mean(),
    "t_statistic": sensitivity_test.statistic,
    "degrees_of_freedom": sensitivity_test.df,
    "p_value": sensitivity_test.pvalue
}])

sensitivity_result_df

,comparison,n_reenactments,difference_pp,t_statistic,degrees_of_freedom,p_value
0,Pooled Dynamic Image - pooled Static Image,378,2.910053,1.550789,377,0.121791


In [11]:
row = sensitivity_result_df.iloc[0]

print(f"Mean difference: {row['difference_pp']:.2f} percentage points")
print(f"t({row['degrees_of_freedom']:.0f}) = {row['t_statistic']:.3f}")
print(f"Two-sided sensitivity-check p-value: {row['p_value']:.12f}")

Mean difference: 2.91 percentage points
t(377) = 1.551
Two-sided sensitivity-check p-value: 0.121791395250


## 7. Participant-Level Heterogeneity Check

This descriptive check examines whether the pooled Dynamic-vs.-Static Image difference is directionally consistent across the eight test participants or is mainly driven by a small number of participants.

For each participant, the table reports the number of reenactments, pooled Static Image accuracy, pooled Dynamic Image accuracy, and the difference $\mathrm{Accuracy}_{Dynamic\ Image} - \mathrm{Accuracy}_{Static\ Image}$.

No participant-level significance test or correction factor is applied. The primary inference remains the reenactment-level analysis above; this section is a heterogeneity and plausibility check.

In [12]:
participant_result_df = (
    analysis_df.reset_index()
    .groupby("participant_id", as_index=False)
    .agg(
        n_reenactments=("reenactment_id_static", "size"),
        static_image_accuracy=("static_image_correct_mean", "mean"),
        dynamic_image_accuracy=("dynamic_image_correct_mean", "mean")
    )
)

participant_result_df["difference_pp"] = 100 * (
    participant_result_df["dynamic_image_accuracy"]
    - participant_result_df["static_image_accuracy"]
)

n_dynamic_better = int((participant_result_df["difference_pp"] > 0).sum())
n_static_better = int((participant_result_df["difference_pp"] < 0).sum())
n_equal = int((participant_result_df["difference_pp"] == 0).sum())
median_participant_difference_pp = participant_result_df["difference_pp"].median()

print(f"Participants favoring Dynamic Image: {n_dynamic_better}/8")
print(f"Participants favoring Static Image:  {n_static_better}/8")
print(f"Participants tied:                    {n_equal}/8")
print(f"Median participant difference (Dynamic - Static Image): {median_participant_difference_pp:.2f} percentage points")

participant_result_df

Participants favoring Dynamic Image: 5/8
Participants favoring Static Image:  2/8
Participants tied:                    1/8
Median participant difference (Dynamic - Static Image): 4.47 percentage points


,participant_id,n_reenactments,static_image_accuracy,dynamic_image_accuracy,difference_pp
0,1,47,0.765957,0.829787,6.382979
1,8,54,0.592593,0.555556,-3.703704
2,10,46,0.489130,0.619565,13.043478
3,13,46,0.804348,0.804348,0.000000
4,15,48,0.687500,0.718750,3.125000
5,18,43,0.802326,0.860465,5.813953
6,23,53,0.698113,0.773585,7.547170
7,27,41,0.780488,0.682927,-9.756098


## 8. Summary and Export

The primary result compares the reported pooled Static and Dynamic Image accuracies while preserving the dependency between Central and Side samples from the same reenactment. The one-sample t-test provides a sensitivity check of the same mean difference. The participant-level breakdown is descriptive and is used to assess directional consistency and heterogeneity across the eight test participants.

In [13]:
summary_df = pd.DataFrame({
    "metric": [
        "Pooled Static Image accuracy",
        "Pooled Dynamic Image accuracy",
        "Dynamic - Static Image difference (pp)",
        "Primary bootstrap CI low (pp)",
        "Primary bootstrap CI high (pp)",
        "Primary bootstrap p-value",
        "Sensitivity t statistic",
        "Sensitivity t-test p-value",
        "Participants favoring Dynamic Image",
        "Participants favoring Static Image",
        "Participants tied",
        "Median participant difference Dynamic - Static Image (pp)"
    ],
    "value": [
        pooled_static_image_accuracy,
        pooled_dynamic_image_accuracy,
        100 * primary_result["difference"],
        100 * primary_result["ci_low"],
        100 * primary_result["ci_high"],
        primary_result["p_value"],
        sensitivity_test.statistic,
        sensitivity_test.pvalue,
        n_dynamic_better,
        n_static_better,
        n_equal,
        median_participant_difference_pp
    ]
})

summary_df

,metric,value
0,Pooled Static Image accuracy,0.698413
1,Pooled Dynamic Image accuracy,0.727513
2,Dynamic - Static Image difference (pp),2.910053
3,Primary bootstrap CI low (pp),-0.793651
4,Primary bootstrap CI high (pp),6.613757
5,Primary bootstrap p-value,0.129288
6,Sensitivity t statistic,1.550789
7,Sensitivity t-test p-value,0.121791
8,Participants favoring Dynamic Image,5.000000
9,Participants favoring Static Image,2.000000


In [14]:
OUTPUT_DIR = Path("statistical_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

analysis_df.to_csv(OUTPUT_DIR / "static_image_vs_dynamic_image_reenactment_table.csv", index=True)
primary_result_df.to_csv(OUTPUT_DIR / "static_image_vs_dynamic_image_primary_result.csv", index=False)
sensitivity_result_df.to_csv(OUTPUT_DIR / "static_image_vs_dynamic_image_sensitivity_ttest.csv", index=False)
participant_result_df.to_csv(OUTPUT_DIR / "static_image_vs_dynamic_image_participant_level_results.csv", index=False)
summary_df.to_csv(OUTPUT_DIR / "static_image_vs_dynamic_image_summary.csv", index=False)

print(f"Results written to: {OUTPUT_DIR.resolve()}")

Results written to: /workspace/repos/emohevrdb-dfer/6_discussion/dynamic-significance-tests/statistical_results
